Week-3 — PySpark Store-Level Insights

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, month, year, sum, avg

spark = SparkSession.builder \
    .appName("Retail Sales Analysis") \
    .getOrCreate()

2. LOAD CLEANED DATA

In [20]:
sales_df = spark.read.csv(
    "cleaned_sales.csv",
    header=True,
    inferSchema=True
)

demand_df = spark.read.csv(
    "cleaned_demand.csv",
    header=True,
    inferSchema=True
)

print("Sales Data")
sales_df.show(5)

print("Demand Data")
demand_df.show(5)

Sales Data
+-------------+-------+-----------+-------------+--------------+----------+--------+----------+----------+----------+---------+-------------+----------+------------+------------------+
|       Region|Country|  Item Type|Sales Channel|Order Priority|Order Date|Order ID| Ship Date|Units Sold|Unit Price|Unit Cost|Total Revenue|Total Cost|Total Profit|   Profit Margin %|
+-------------+-------+-----------+-------------+--------------+----------+--------+----------+----------+----------+---------+-------------+----------+------------+------------------+
|         Asia|  India|Electronics|       Online|             H|2024-01-05|   10001|2024-01-10|       120|       250|      180|        30000|     21600|        8400|28.000000000000004|
|         Asia|  India|  Furniture|      Offline|             M|2024-01-08|   10002|2024-01-14|        75|       400|      320|        30000|     24000|        6000|              20.0|
|       Europe|Germany|   Clothing|       Online|             L|

 Use pandas to clean missing values, correct data types

In [21]:
sales_df = sales_df.withColumn(
    "Month", month(col("Order Date"))
).withColumn(
    "Year", year(col("Order Date"))
)


 4. STORE LEVEL REVENUE

In [22]:
store_revenue = sales_df.groupBy(
    "Region", "Month"
).agg(
    sum("Total Revenue").alias("Monthly Revenue")
)

print("\nStore Monthly Revenue")
store_revenue.show()


Store Monthly Revenue
+-------------+-----+---------------+
|       Region|Month|Monthly Revenue|
+-------------+-----+---------------+
|       Africa|    1|           6000|
|North America|    1|          45000|
|       Europe|    1|          10000|
|       Europe|    2|          30000|
|         Asia|    2|          32400|
|North America|    2|           4500|
|         Asia|    1|          60000|
|       Africa|    2|           5600|
+-------------+-----+---------------+



5. AVERAGE MONTHLY REVENUE

In [23]:
avg_monthly_revenue = store_revenue.groupBy("Region") \
    .agg(avg("Monthly Revenue").alias("Avg Monthly Revenue"))

print("\nAverage Monthly Revenue per Region")
avg_monthly_revenue.show()


Average Monthly Revenue per Region
+-------------+-------------------+
|       Region|Avg Monthly Revenue|
+-------------+-------------------+
|       Europe|            20000.0|
|       Africa|             5800.0|
|North America|            24750.0|
|         Asia|            46200.0|
+-------------+-------------------+



In [29]:
# 6. FIND UNDERPERFORMING PRODUCTS
# Rule:
# Low Units Sold + Low Profit
# ----------------------------
underperforming_products = sales_df.filter(
    (col("Units Sold") < 100) &
    (col("Total Profit") < 5000)
)

print("\nUnderperforming Products")
underperforming_products.select(
    "Item Type",
    "Region",
    "Units Sold",
    "Total Profit"
).show()



Underperforming Products
+---------+------+----------+------------+
|Item Type|Region|Units Sold|Total Profit|
+---------+------+----------+------------+
+---------+------+----------+------------+



In [30]:
# 7. STORE PERFORMANCE RANKING
# ----------------------------
store_performance = sales_df.groupBy("Region") \
    .agg(sum("Total Revenue").alias("Total Store Revenue")) \
    .orderBy(col("Total Store Revenue").desc())

print("\nStore Performance Ranking")
store_performance.show()


Store Performance Ranking
+-------------+-------------------+
|       Region|Total Store Revenue|
+-------------+-------------------+
|         Asia|              92400|
|North America|              49500|
|       Europe|              40000|
|       Africa|              11600|
+-------------+-------------------+



In [31]:
# 8. SAVE OUTPUT FILES
# ----------------------------
underperforming_products.write.csv(
    "underperforming_products",
    header=True,
    mode="overwrite"
)

store_performance.write.csv(
    "store_summary",
    header=True,
    mode="overwrite"
)

print("\n✅ Week-3 Outputs Saved Successfully!")


✅ Week-3 Outputs Saved Successfully!
